# 1 · Quickstart

The Zn/MnO₂ cell model from Python: a Zn anode, an open electrolyte region (where a pH probe sits in the experimental cell), a separator and a porous MnO₂ cathode, in 2 M ZnSO₄ + 0.05 M MnSO₄.

The cathode reacts by Zn insertion (R3), by dissolution and deposition of Zn_zMnO₂ (R2) and by dissolution of pristine MnO₂ (R1); zinc hydroxide sulfate (ZHS) precipitates when the solution is supersaturated. The pH comes from the solution equilibria at every point. See `docs/model.md`.

The default parameters are illustrative, not fitted to a cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

## A discharge

`Params()` holds every input (`docs/parameters.md`); the protocol is a string (`docs/protocol.md`). Currents are in mA/g of active material, positive for discharge.

In [ ]:
p = Params(steps="cc I=100 Vmin=1.0")
r = run(p)
print(r.exit_reason, f"{r.column('mAhg')[-1]:.1f} mAh/g after {r.steps} time steps")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(r.column("mAhg"), r.column("V"))
ax[0].set(xlabel="capacity [mAh/g]", ylabel="cell voltage [V vs Zn]")
for c in ("pH_cath", "pH_probe", "pH_anode"):
    ax[1].plot(r.column("mAhg"), r.column(c), label=c)
ax[1].set(xlabel="capacity [mAh/g]", ylabel="pH"); ax[1].legend();

The first plateau is Zn insertion; the pH rises as R1 and R2 consume protons. Near 145 mAh/g the host is full, R2 dissolution takes over at a lower voltage, and ZHS precipitates, holding the cathode pH at ZHS saturation (about 5.4).

## Which reaction carries the current

In [ ]:
q = r.column("mAhg")
for c, lab in (("i_R1_mAg", "R1 pristine MnO2"), ("i_R2_mAg", "R2 ZMO dissolution/deposition"), ("i_R3_mAg", "R3 Zn insertion")):
    plt.plot(q, -r.column(c), label=lab)
plt.xlabel("capacity [mAh/g]"); plt.ylabel("cathodic current [mA/g]"); plt.legend();

## Discharge, then charge

`end_on_cutoff=False` lets a cutoff end only its step, so the protocol continues with the charge.

In [ ]:
r = run(Params(steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False))
q = r.column("mAhg")
print(r.exit_reason, f"discharged {q.max():.1f} mAh/g, {q.max() - q[-1]:.1f} mAh/g back on charge")
plt.plot(q, r.column("V")); plt.xlabel("capacity [mAh/g]"); plt.ylabel("V");

## GITT

A pulse and a rest, repeated until the cutoff (`end_on_cutoff` is true by default). This takes a minute or two.

In [ ]:
r = run(Params(steps="cc I=50 t=720; rest t=1800", cycles=1000, write_interval=30.0))
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(r.column("t_h"), r.column("V"), lw=0.8); ax[0].set(ylabel="V")
ax[1].plot(r.column("t_h"), r.column("pH_cath"), lw=0.8, label="cathode")
ax[1].plot(r.column("t_h"), r.column("pH_probe"), lw=0.8, label="probe")
ax[1].set(xlabel="time [h]", ylabel="pH"); ax[1].legend();

## Input files

The command line reads the same inputs from a namelist file: `python -m znmno2_model input/default.nml`. From Python:

In [ ]:
from pathlib import Path
from znmno2_model.namelist import load

root = Path.cwd() if (Path.cwd() / "input").exists() else Path.cwd().parent
p, extra = load(root / "input" / "default.nml")
print(p == Params(), extra)